# with stable diffusion v1-5

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
from diffusers import UNet2DConditionModel, AutoencoderKL, DDPMScheduler
import matplotlib.pyplot as plt
from utils import im2tensor, viewimage

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"

print(f"Loading {model_id}...")

# 1. Load the individual components


# run only once
download = False
if download:
    unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet").to(device)
    vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae").to(device)
    scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")
    vae.save_pretrained("./models/stable-diffusion-v1-5/vqvae")
    unet.save_pretrained("./models/stable-diffusion-v1-5/unet")
    scheduler.save_pretrained("./models/stable-diffusion-v1-5/scheduler")
    
else:
    vae = AutoencoderKL.from_pretrained("./models/stable-diffusion-v1-5/vqvae").to(device)
    unet = UNet2DConditionModel.from_pretrained("./models/stable-diffusion-v1-5/unet").to(device)
    scheduler = DDPMScheduler.from_pretrained("./models/stable-diffusion-v1-5/scheduler")

# 2. Count and print the number of parameters
unet_params = sum(p.numel() for p in unet.parameters())
vae_params = sum(p.numel() for p in vae.parameters())

print("-" * 30)
print(f"UNet Parameters: {unet_params:,}")
print(f"VAE Parameters:  {vae_params:,}")
print(f"Total SD v1.5 Parameters: {unet_params + vae_params:,}")
print("-" * 30)

# 3. Freeze the models to save massive amounts of memory and compute
unet.eval()
vae.eval()
for param in unet.parameters():
    param.requires_grad = False
for param in vae.parameters():
    param.requires_grad = False

In [ ]:
MODE = "inpainting"

idx = 469
x0 = im2tensor(plt.imread('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png'), device=device)
x0

imgshape = x0.shape
imgshape_latent = (1, unet.in_channels, unet.sample_size, unet.sample_size)

sigma_noise = 0.01

if MODE == "inpainting":
    h = imgshape[2]
    w = imgshape[3]
    hcrop, wcrop = h//2, w//2
    corner_top, corner_left = h//4, int(0.45*w)
    mask = torch.ones(imgshape, device=device)
    mask[:,:,corner_top:corner_top+hcrop,corner_left:corner_left+wcrop] = 0

    def linear_operator(x):
        x = x*mask
        return(x)

    x_true = x0.clone()

    y = linear_operator(x_true.clone()) + sigma_noise * mask * torch.randn_like(x_true)
    vis_y = y    
    
viewimage(torch.cat((x_true, vis_y), dim=3), titre='Ground truth and measurements', displayfilename=False)

In [ ]:
# --- Scheduling Setup ---
num_inference_steps = 20
scheduler.set_timesteps(num_inference_steps=num_inference_steps)

true_timesteps = scheduler.timesteps.to(device)
alpha_bar = scheduler.alphas_cumprod[true_timesteps].to(device)
alpha_bar_prev = torch.cat([alpha_bar[1:], torch.tensor([1.0], device=device)])
alpha = alpha_bar / alpha_bar_prev
beta = 1.0 - alpha

# SD 1.5 expects latents of shape (batch, 4, 64, 64) for a 512x512 image
imgshape_latent = (1, 4, 32, 32)
z = torch.randn(imgshape_latent).to(device)
scaling_factor = vae.config.scaling_factor

# Empty prompt embedding since we aren't using text conditioning
# Shape: (batch_size, sequence_length, hidden_size) for SD 1.5
empty_text_embeds = torch.zeros((1, 77, 768), device=device)

# --- Generation Loop ---
for i, true_t in enumerate(tqdm(true_timesteps, total=num_inference_steps)):
    
    # 1. Fast U-Net pass (NO GRADIENTS)
    with torch.no_grad():
        # U-Net needs the timestep as a 1D tensor
        t_input = torch.tensor([true_t], device=device)
        noise_pred = unet(z, t_input, encoder_hidden_states=empty_text_embeds).sample.detach()
        
    score_pred = - noise_pred / (1 - alpha_bar[i]) ** 0.5
    
    # 2. START gradient tracking only for z
    z = z.detach().requires_grad_(True)
    
    # Calculate predicted original latent
    z_0_hat = (1 / alpha_bar[i] ** 0.5) * (z + (1 - alpha_bar[i]) * score_pred)
    
    # 3. Measurement Update using VAE Decoder
    # ALWAYS unscale before decoding
    z_0_hat_unscaled = z_0_hat / scaling_factor
    x0_hat = vae.decode(z_0_hat_unscaled).sample
    
    # Calculate measurement loss 
    loss = ((linear_operator(x0_hat) - y) ** 2).sum() 
        
    # Calculate gradients
    grad = torch.autograd.grad(loss, z)[0]
    zeta = 0.1 / (np.sqrt(loss.item()) + 1e-8)

    # 4. DDPM Posterior Mean Step
    coef1 = (torch.sqrt(alpha[i]) * (1 - alpha_bar_prev[i])) / (1 - alpha_bar[i])
    coef2 = (torch.sqrt(alpha_bar_prev[i]) * beta[i]) / (1 - alpha_bar[i])
    z_prim = coef1 * z + coef2 * z_0_hat 
    
    # Add variance if not the last step
    if i < len(true_timesteps) - 1:
        z_prim += torch.sqrt(beta[i]) * torch.randn_like(z)

    # Apply measurement gradient
    z = z_prim - zeta * grad
    
    # TODO: Add Gluing Term Here

# --- Final Output ---
with torch.no_grad():
    z_final_unscaled = z.detach() / scaling_factor
    final_img = vae.decode(z_final_unscaled).sample

# Process final_img for viewing (clip, scale to 0-255, convert to NHWC)

In [ ]:
viewimage(torch.cat((x_true, final_img, vis_y), dim=3), titre="Final_Results")